# 04 — Bayesian vs Frequentist Comparison

Run a vanilla sklearn logistic regression on the same data and compare. The story isn't whether Bayesian wins on AUC — it's what it adds that frequentist can't provide.

In [ ]:
import numpy as np
import pandas as pd
import arviz as az
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss, roc_curve
from sklearn.calibration import calibration_curve
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import warnings; warnings.filterwarnings("ignore")

df = pd.read_csv("../data/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df["Churn"] = (df["Churn"] == "Yes").astype(int)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df = df.dropna()
df = pd.get_dummies(df, columns=["Contract", "InternetService"], drop_first=False)
df.columns = df.columns.str.replace(" ", "_").str.replace("-", "_")

features = [
    "tenure", "MonthlyCharges", "TotalCharges",
    "Contract_Month_to_month", "Contract_Two_year",
    "InternetService_Fiber_optic", "SeniorCitizen",
]

X = StandardScaler().fit_transform(df[features])
y = df["Churn"].values
idx = np.arange(len(y))
idx_tr, idx_te = train_test_split(idx, test_size=0.2, random_state=42)
X_tr, X_te = X[idx_tr], X[idx_te]
y_tr, y_te = y[idx_tr], y[idx_te]
print(f"Test set: {len(y_te)} customers, {y_te.sum()} churners")

## Frequentist baseline

In [ ]:
freq = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
freq_proba = freq.predict_proba(X_te)[:, 1]
print("Frequentist fitted.")

## Bayesian posterior predictive on test set

In [ ]:
import os

trace_path = "../outputs/trace.nc"
if not os.path.exists(trace_path):
    raise FileNotFoundError(
        "trace.nc not found. Run 02_bayesian_model.ipynb first to generate it."
    )

trace      = az.from_netcdf(trace_path)
betas_flat = trace.posterior["betas"].values.reshape(-1, 7)
alpha_flat = trace.posterior["alpha"].values.reshape(-1)

logit_s   = alpha_flat[:, None] + betas_flat @ X_te.T
p_samples = 1 / (1 + np.exp(-logit_s))   # (4000, N_te)
bayes_proba = p_samples.mean(axis=0)
bayes_std   = p_samples.std(axis=0)
print("Bayesian posterior predictive computed.")

## Metrics

In [ ]:
print(f"{'Metric':<25} {'Frequentist':>14} {'Bayesian':>12}")
print("-" * 53)
print(f"{'AUC':<25} {roc_auc_score(y_te, freq_proba):>14.4f} {roc_auc_score(y_te, bayes_proba):>12.4f}")
print(f"{'Brier score (lower=better)':<25} {brier_score_loss(y_te, freq_proba):>14.4f} {brier_score_loss(y_te, bayes_proba):>12.4f}")
print()
print("Key insight: AUC is nearly identical.")
print("The Bayesian edge is uncertainty quantification, not raw accuracy.")

## Plots: ROC, Calibration, Uncertainty

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ROC
ax = axes[0]
for proba, label, col, ls in [
    (freq_proba,  "Frequentist LR", "#E07B39", "--"),
    (bayes_proba, "Bayesian LR",    "#378ADD", "-"),
]:
    fpr, tpr, _ = roc_curve(y_te, proba)
    auc = roc_auc_score(y_te, proba)
    ax.plot(fpr, tpr, color=col, linestyle=ls, linewidth=2, label=f"{label} (AUC={auc:.4f})")
ax.plot([0,1],[0,1],"k:",alpha=0.4, label="Random")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve", fontweight="bold")
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Calibration
ax = axes[1]
for proba, label, col, ls in [
    (freq_proba,  "Frequentist LR", "#E07B39", "--"),
    (bayes_proba, "Bayesian LR",    "#378ADD", "-"),
]:
    fp, mp = calibration_curve(y_te, proba, n_bins=10)
    ax.plot(mp, fp, color=col, linestyle=ls, linewidth=2, marker="o", markersize=4, label=label)
ax.plot([0,1],[0,1],"k:",alpha=0.5, label="Perfect calibration")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Fraction of positives")
ax.set_title("Calibration Curve", fontweight="bold")
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Uncertainty — Bayesian only
ax = axes[2]
ax.hist(bayes_std[y_te==0], bins=40, alpha=0.65, color="#378ADD",
        density=True, label="Actual: no churn", edgecolor="white", linewidth=0.3)
ax.hist(bayes_std[y_te==1], bins=40, alpha=0.65, color="#D7263D",
        density=True, label="Actual: churned", edgecolor="white", linewidth=0.3)
ax.set_xlabel("Posterior std of churn probability")
ax.set_ylabel("Density")
ax.set_title("Bayesian Prediction Uncertainty\nby Actual Outcome (no frequentist equivalent)",
             fontweight="bold")
ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)

plt.suptitle("Frequentist vs Bayesian — same accuracy, richer inference",
             fontsize=12, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## Summary

| | Frequentist LR | Bayesian LR |
|---|---|---|
| AUC | ~0.82 | ~0.82 |
| Calibration (Brier) | Similar | Marginally better |
| Per-prediction uncertainty | ✗ | ✓ |
| Credible intervals on coefficients | ✗ | ✓ |
| Principled incorporation of prior knowledge | ✗ | ✓ |

**When to choose Bayesian:** small data, available prior knowledge, or when the cost of acting on an uncertain prediction is high.

In [ ]:
# To regenerate requirements.txt, run this in your terminal:
# pip freeze > requirements.txt